In [1]:
import pandas as pd
from scipy.stats import chi2_contingency

In [2]:
df = pd.read_excel('ETL_mentions.xlsx')

In [3]:
df.head()

,Session,Code_académie,Libellé_académie,Sexe,Voie,Mention,TOTAL
0,2024,16,TOULOUSE,GARCONS,BAC GENERAL,TB_F,117
1,2024,16,TOULOUSE,GARCONS,BAC GENERAL,TB,789
2,2024,16,TOULOUSE,GARCONS,BAC GENERAL,B,1660
3,2024,16,TOULOUSE,GARCONS,BAC GENERAL,AB,2282
4,2024,16,TOULOUSE,GARCONS,BAC GENERAL,Sans_mention,2155


In [4]:
import pandas as pd
from scipy.stats import chi2_contingency

# 1. Création de la matrice en sommant la colonne 'Valeur'
# C'est ici qu'on dit à Python : "Additionne le nombre d'élèves"
matrice_acad = pd.crosstab(index=df['Libellé_académie'], 
                           columns=df['Mention'], 
                           values=df['TOTAL'], 
                           aggfunc='sum').fillna(0)

# 2. Exécution du test sur les vrais volumes
from scipy.stats import chi2_contingency
stat, p, dof, expected = chi2_contingency(matrice_acad)

print(f"Statistique de test : {stat}")
print(f"P-value : {p}")

Statistique de test : 42410.2382348955
P-value : 0.0


In [5]:
import plotly.express as px

# 1. On calcule le total d'élèves par Académie pour normaliser
df_sum = df.groupby('Libellé_académie')['TOTAL'].transform('sum')
df['Pourcentage'] = (df['TOTAL'] / df_sum) * 100

# 2. On regroupe pour le graphique
df_plot = df.groupby(['Libellé_académie', 'Mention'])['Pourcentage'].sum().reset_index()

# 3. Création du graphique avec facettes
fig = px.bar(df_plot, 
             x="Libellé_académie", 
             y="Pourcentage", 
             color="Mention",
             facet_row="Mention",
             title="Profil de Performance : % de chaque Mention par Académie",
             labels={"Pourcentage": "% de l'effectif académique"},
             height=1200)

# 4. Ajustements cruciaux
fig.update_yaxes(matches=None, ticksuffix="%") # Échelles indépendantes pour voir les nuances du TB
fig.update_xaxes(tickangle=45, categoryorder='total descending')
fig.update_layout(showlegend=False)

fig.show()

In [6]:
import plotly.express as px

# 1. Préparation des données (Somme des effectifs par académie et mention)
df_plot = df.groupby(['Libellé_académie', 'Mention'])['TOTAL'].sum().reset_index()

# 2. Création du graphique avec facettes
fig = px.bar(df_plot, 
             x="Libellé_académie", 
             y="TOTAL", 
             color="Libellé_académie",
             facet_row="Mention",  # Une ligne par mention
             title="Comparaison des Académies par Type de Mention",
             labels={"TOTAL": "Nombre d'élèves"},
             height=1200) # On augmente la hauteur pour que chaque ligne soit lisible

# 3. Ajustements pour la lisibilité
fig.update_layout(showlegend=False) # La légende est inutile ici car l'axe X suffit
fig.update_xaxes(tickangle=45) # Rotation des noms d'académies

# Pour que chaque facette ait sa propre échelle d'axe Y 
# (sinon les mentions TB seront minuscules par rapport aux "Sans mention")
fig.update_yaxes(matches=None) 

fig.show()

In [7]:
import plotly.express as px

# On calcule le total d'élèves par académie pour normaliser
df_total = df.groupby('Libellé_académie')['TOTAL'].transform('sum')
df['Pourcentage'] = (df['TOTAL'] / df_total) * 100

# Graphique en barres 100% empilées
fig = px.bar(df, 
             x="Libellé_académie", 
             y="Pourcentage", 
             color="Mention",
             title="Profil de Performance : Répartition des Mentions en % par Académie",
             labels={"Pourcentage": "% des élèves"},
             category_orders={"Mention": ["TB", "B", "AB", "Sans_mention", "Refusés"]},
             barmode="relative")

fig.update_layout(yaxis_ticksuffix="%", xaxis={'categoryorder':'total descending'})
fig.show()